In [1]:
!pip install pyspark
from pyspark import SparkContext, SparkConf
conf = SparkConf().setAppName("appName").setMaster("local[*]")
sc = SparkContext(conf=conf)
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.0-py2.py3-none-any.whl size=311317145 sha256=d5d894a8bbe61027020f35ee8d3679aa00dd2cc6a2798e253897beec9349c508
  Stored in directory: /root/.cache/pip/wheels/7b/1b/4b/3363a1d04368e7ff0d408e57ff57966fcdf00583774e761327
Successfully built pyspark


In [11]:
inputfile = spark.read.csv("Movies.csv", header= True)
inputfile.show()

+----+------+--------------------+-------+--------------------+-----------------+--------------------+----------+------+-------------------+
|Year|Length|               Title|  Genre|               Actor|          Actress|            Director|Popularity|Awards|              Image|
+----+------+--------------------+-------+--------------------+-----------------+--------------------+----------+------+-------------------+
|1990|   111|Tie Me Up! Tie Me...| Comedy|     BanderasAntonio|    AbrilVictoria|      AlmodóvarPedro|        68|    No|   NicholasCage.png|
|1991|   113|          High Heels| Comedy|          BoséMiguel|    AbrilVictoria|      AlmodóvarPedro|        68|    No|   NicholasCage.png|
|1983|   104|        Dead ZoneThe| Horror|   WalkenChristopher|      AdamsBrooke|     CronenbergDavid|        79|    No|   NicholasCage.png|
|1979|   122|                Cuba| Action|         ConnerySean|      AdamsBrooke|       LesterRichard|         6|    No|    seanConnery.png|
|1978|    94|

In [12]:
awards_winners = inputfile.filter((inputfile.Awards == "Yes") & (inputfile.Genre == "Action"))
awards_winners.select("Title", "Year", "Director").show()

+-----+----+--------+
|Title|Year|Director|
+-----+----+--------+
+-----+----+--------+



In [20]:
from pyspark.sql.functions import col
award_actors = inputfile.filter(col("Awards") == "Yes").select("Actor").distinct()
for row in award_actors.collect():
    actor = row["Actor"]
    actor_movies = inputfile.filter((col("Awards") == "Yes") & (col("Actor") == actor))
    actor_movies.select("Title", "Director").show()

+----------+--------------+
|     Title|      Director|
+----------+--------------+
|AssaultThe|RademakersFons|
+----------+--------------+

+--------------------+------------+
|               Title|    Director|
+--------------------+------------+
|             Airport|SeatonGeorge|
|Come BackLittle S...|  MannDaniel|
+--------------------+------------+

+---------+----------+
|    Title|  Director|
+---------+----------+
|Norma Rae|RittMartin|
+---------+----------+

+--------------------+---------------+
|               Title|       Director|
+--------------------+---------------+
|Garden of the Fin...|De SicaVittorio|
+--------------------+---------------+

+---------------+------------------+
|          Title|          Director|
+---------------+------------------+
|Last EmperorThe|BertolucciBernardo|
+---------------+------------------+

+-------------------+---------------+
|              Title|       Director|
+-------------------+---------------+
|Reversal of Fortune|Schroeder

In [22]:
non_award_popular_movies = inputfile.filter(col("Awards") == "No").sort(col("Popularity").desc()).limit(10)
non_award_popular_movies.show()

+----+------+--------------------+-------+-----------------+------------------+--------------------+----------+------+----------------+
|Year|Length|               Title|  Genre|            Actor|           Actress|            Director|Popularity|Awards|           Image|
+----+------+--------------------+-------+-----------------+------------------+--------------------+----------+------+----------------+
|1989|   104|        Tango & Cash| Action|StalloneSylvester|       HatcherTeri|  KonchalovskyAndrei|         9|    No|NicholasCage.png|
|1985|   124|              Plenty|  Drama|     DanceCharles|       StreepMeryl|        SchepisiFred|         9|    No| merylStreep.png|
|1989|    83|Masque of the Red...| Horror|    MacNeePatrick|         HoakClare|          BrandLarry|         9|    No|NicholasCage.png|
|1987|   104|           Lionheart| Action|       StoltzEric|  BarrymoreDeborah|SchaffnerFranklin J.|         9|    No|NicholasCage.png|
|1934|    80|        Judge Priest|  Drama|      

In [24]:
old_unpopular_movies = inputfile.filter((col("Year") < 1980) & (col("Popularity").isNotNull())).sort(col("Popularity")).limit(10)
old_unpopular_movies.show()

+----+------+--------------------+--------+-----------------+----------------+----------------+----------+------+------------------+
|Year|Length|               Title|   Genre|            Actor|         Actress|        Director|Popularity|Awards|             Image|
+----+------+--------------------+--------+-----------------+----------------+----------------+----------+------+------------------+
|1968|   113|             Shalako|Westerns|      ConnerySean|  BardotBrigitte|   DmytrykEdward|         0|    No|brigitteBardot.png|
|1970|   137|             Airport|   Drama|    LancasterBurt|BissetJacqueline|    SeatonGeorge|         0|   Yes| burtLancaster.png|
|1930|    92|       Anna Christie|   Drama|  BickfordCharles|      GarboGreta|   BrownClarence|         0|    No|    gretaGarbo.png|
|1976|   128|  Shout at the Devil|  Action|        MarvinLee|  ParkinsBarbara|    HuntPeter R.|         0|    No|  NicholasCage.png|
|1953|   120|   Tales of Tomorrow|  Horror|     KarloffBoris|        

In [25]:
genre_avg_length = inputfile.groupBy("Genre").agg({"Length": "avg"})
genre_avg_length.show()

+---------------+------------------+
|          Genre|       avg(Length)|
+---------------+------------------+
|          Crime|              66.0|
|        Romance|             127.0|
|      Adventure|             119.0|
|           null|             120.5|
|          Drama|113.30455259026688|
|            War|         116.90625|
|        Fantasy|             102.0|
|        Mystery|103.00990099009901|
|          Music|100.48780487804878|
|Science Fiction|106.47368421052632|
|         Horror| 93.92727272727272|
|          Short|              40.0|
|        Western|  93.0091743119266|
|         Comedy| 96.50540540540541|
|         Action|             104.5|
|       Westerns|             124.8|
+---------------+------------------+



In [26]:
comedy_pairs = inputfile.filter(col("Genre") == "Comedy").groupBy("Actor", "Actress").count().filter(col("count") > 3)
comedy_pairs.show()

+-------------+----------------+-----+
|        Actor|         Actress|count|
+-------------+----------------+-----+
|ChapmanGraham|            null|    6|
| TracySpencer|HepburnKatharine|    6|
|  MurphyEddie|            null|    4|
|   CleeseJohn|            null|    4|
| SellersPeter|            null|   11|
|   AllenWoody|     KeatonDiane|    5|
|  MartinSteve|            null|    4|
|WilliamsRobin|            null|    5|
+-------------+----------------+-----+



In [27]:
comedy_actors = inputfile.filter(col("Genre") == "Comedy").select("Actor").distinct()
drama_actors = inputfile.filter(col("Genre") == "Drama").select("Actor").distinct()
comedy_drama_actors = comedy_actors.intersect(drama_actors)
comedy_drama_actors.show()

+---------------+
|          Actor|
+---------------+
|    WillisBruce|
|    IronsJeremy|
|  EastwoodClint|
|    ConnerySean|
|   TracySpencer|
|     NelsonJudd|
|    AielloDanny|
|    MooreDudley|
|     QuinnAidan|
| AdolphsonEdvin|
|     FinchPeter|
|   BrandoMarlon|
|  HopkinsHarold|
|     NewmanPaul|
|   BeattyWarren|
|   BoyerCharles|
|      CaanJames|
|HowellC. Thomas|
|   SheenCharlie|
|     RogersWill|
+---------------+
only showing top 20 rows



In [28]:
comedy_or_drama_actors = inputfile.filter((col("Genre") == "Comedy") | (col("Genre") == "Drama")).select("Actor").distinct()
comedy_or_drama_actors.show()

+---------------+
|          Actor|
+---------------+
|     BoséMiguel|
|   CottenJoseph|
|     DillonMatt|
|  KeatonMichael|
| ShimuraTakashi|
|   LintDerek De|
|    WillisBruce|
|  LancasterBurt|
|    RomeroCesar|
|   JourdanLouis|
|  ModineMatthew|
|    JaglomHenry|
|       DavisGuy|
|    BridgesBeau|
|    BakulaScott|
|    DoranJohnny|
|CapolicchioLino|
|     FondaPeter|
|       LeeKarla|
| TownsendRobert|
+---------------+
only showing top 20 rows



In [29]:
non_comedy_actors = inputfile.filter(col("Genre") != "Comedy").select("Actor").distinct()
non_comedy_actors.show()

+---------------+
|          Actor|
+---------------+
|   CottenJoseph|
|       BrownTom|
|     DillonMatt|
| ShimuraTakashi|
|   LintDerek De|
|  LancasterBurt|
|    WillisBruce|
|    RomeroCesar|
|  StockwellDean|
|  ModineMatthew|
|    UrichRobert|
|       DavisGuy|
|    BridgesBeau|
|    KattWilliam|
|  EnglundRobert|
|      PriceMarc|
|CapolicchioLino|
|     FondaPeter|
| TownsendRobert|
|  ChesneyArthur|
+---------------+
only showing top 20 rows



In [30]:
actor_rankings = inputfile.groupBy("Actor").agg({"Popularity": "mean", "Popularity": "max", "Popularity": "min"})
actor_rankings.show()

+------------------+---------------+
|             Actor|min(Popularity)|
+------------------+---------------+
|              null|             32|
|        AbelAlfred|             49|
|  AbrahamF. Murray|              6|
|    AdolphsonEdvin|             26|
|       AherneBrian|             57|
|     AhlstedtBörje|             81|
|       AielloDanny|             20|
|         AkanTarik|             53|
|    AlbaicínRafael|             55|
|      AlbertEdward|             82|
|          AldaAlan|             12|
|         AllenBill|             75|
|        AllenWoody|             12|
|     AlterioHector|             39|
|         AmecheDon|             45|
|     AndersonKevin|             53|
|   AnderssonWiktor|             66|
|AngladeJean-Hughes|             71|
|       AnneseFrank|             45|
|        ApfelOscar|             66|
+------------------+---------------+
only showing top 20 rows



In [45]:
from pyspark.sql.functions import col, lit

movie_counts_decade = (inputfile
                      .withColumn("Decade", ((col("Year").cast("int") - 1960) // lit(10).cast("int")) * lit(10).cast("int") + 1960)
                      .groupBy("Decade")
                      .count()
                      .orderBy("Decade"))

movie_counts_decade.show()

TypeError: ignored

In [32]:
inputfile = spark.read.csv("Movies.csv", header=True)

movies_per_year = inputfile.groupBy("Year").count().orderBy("Year")
movies_per_year.show()

+----+-----+
|Year|count|
+----+-----+
|1920|    1|
|1923|    1|
|1924|    3|
|1925|    1|
|1926|    4|
|1927|    3|
|1928|    5|
|1929|    5|
|1930|    3|
|1931|    9|
|1932|    8|
|1933|    3|
|1934|    4|
|1935|    8|
|1936|    6|
|1937|    9|
|1938|    8|
|1939|   11|
|1940|   11|
|1941|    7|
+----+-----+
only showing top 20 rows



In [33]:
inputfile = spark.read.csv("Movies.csv", header=True)

long_movies = inputfile.filter(inputfile.Length > 100)
movies_per_year_genre = long_movies.groupBy("Year", "Genre").count().orderBy("Year", "Genre")
movies_per_year_genre.show()

+----+---------------+-----+
|Year|          Genre|count|
+----+---------------+-----+
|1920|          Drama|    1|
|1924|          Drama|    2|
|1925|          Drama|    1|
|1926|         Action|    1|
|1926|          Drama|    1|
|1926|Science Fiction|    1|
|1928|          Drama|    2|
|1928|            War|    1|
|1929|          Drama|    1|
|1931|        Western|    2|
|1932|         Action|    2|
|1932|          Drama|    1|
|1933|          Drama|    1|
|1935|        Western|    1|
|1936|          Drama|    1|
|1938|          Drama|    1|
|1938|        Western|    2|
|1939|         Action|    1|
|1939|         Comedy|    1|
|1939|        Western|    3|
+----+---------------+-----+
only showing top 20 rows



In [34]:
inputfile = spark.read.csv("Movies.csv", header=True)

movies_before_1990 = inputfile.filter(inputfile.Year < 1990)
movies_before_1990.sort("Title").show()

+----+------+--------------------+---------------+--------------------+-----------------+-------------------+----------+------+-------------------+
|Year|Length|               Title|          Genre|               Actor|          Actress|           Director|Popularity|Awards|              Image|
+----+------+--------------------+---------------+--------------------+-----------------+-------------------+----------+------+-------------------+
|1968|   139|2001: A Space Ody...|Science Fiction|          DulleaKeir|   TyzackMargaret|     KubrickStanley|        83|    No|   NicholasCage.png|
|1982|    92|             48 Hrs.|         Action|           NolteNick|   O'TooleAnnette|         HillWalter|        67|    No|   NicholasCage.png|
|1963|   138|               8 1/2|          Drama| MastroianniMarcello| CardinaleClaudia|    FelliniFederico|        80|   Yes|   NicholasCage.png|
|1966|    95|A Big Hand for th...|         Comedy|          FondaHenry|   WoodwardJoanne|        CookFielder|   

In [37]:
from pyspark.sql.functions import length

inputfile = spark.read.csv("Movies.csv", header=True)
long_title_movies = inputfile.filter(length(inputfile.Title) > 50)
long_title_movies.show()

+----+------+--------------------+------+----------+--------------+--------+----------+------+----------------+
|Year|Length|               Title| Genre|     Actor|       Actress|Director|Popularity|Awards|           Image|
+----+------+--------------------+------+----------+--------------+--------+----------+------+----------------+
|1979|    90|Fawlty TowersGour...|Comedy|CleeseJohn|ScalesPrunella|    null|        46|    No|NicholasCage.png|
+----+------+--------------------+------+----------+--------------+--------+----------+------+----------------+

